In [1]:
# --- Rutas del proyecto -------------------------------------------------
# Definición única en projects/mayanlab/mayanlab/paths.py. Funciona desde
# cualquier directorio de trabajo: no hay rutas relativas en este notebook.
from pathlib import Path

from mayanlab.paths import (
    AUDIO, ANNOTATIONS, DATA, MANIFESTS, SEGMENTS, TRANSCRIPTS, WORK,
    NARRACIONES_MP3, NARRACIONES_TRANSCRIPTS, CLEAN_TEXT, ensure, project,
)

P = project("corpus")
FIGURES = P.figures
ensure(FIGURES)

# Dataset con timestamps

En esta notebook obtendremos un dataset en formato json que nos permitirá recrear todo el dataset con un script de python

Como recordatorio, el dataset está conformado por dos datasets diferentes. Uno de 1 hora, proveniente de segmentos de YouTube y otro de 3 horas, proveniente de las Narraciones Mayas de Campeche.

Dividiremos esta notebook en los siguientes pasos:

- Paso 0. Importar librerías y nuestro dataset de huggingface
- Paso 1. Para el dataset de 1h ---- "1h-raw-data.csv"
- Paso 2. Para el dataset de 3h. A partir de "3h-transcripts.csv" asociar las transcripciones de huggingface con los timestamps de este csv.
- Paso 3. Combinar ambos datasets y exportarlos en formato json.

## Paso 0

In [2]:
from huggingface_hub import notebook_login
import json
import re
from datasets import Audio, Dataset, Features, Value, load_dataset

RAW_DATA_1H = ANNOTATIONS / "1h-raw-data.csv"
RAW_DATA_3H = ANNOTATIONS / "3h-transcripts.csv"
SPK_META_PATH = ANNOTATIONS / "spk_metadata.csv"

# Puente utt_id (spk-based, HF) <-> filename (id crudo url-derivado)
BRIDGE_PATH = MANIFESTS / "dataset.csv"
# Fuentes ya curadas de spk_001-018 (YouTube + grabaciones propias)
SEGMENTS_JSON_IN = ANNOTATIONS / "corrected_segments_out.json"
# Salida: manifiesto de procedencia de audio (3 grupos, SIN texto)
OUT_JSON = MANIFESTS / "source_segments.json"

# URL publica base de las narraciones mayas de Campeche (INALI). Solo se cita la fuente del AUDIO.
INALI_BASE = "https://site.inali.gob.mx/publicaciones/audios/narraciones_mayas_campeche"

REPO_ID = "mau-cr/mayan-voice"

In [3]:
notebook_login()

In [4]:
hf_ds = load_dataset(REPO_ID)

In [5]:
ds = hf_ds["train"]

In [6]:
ds

Dataset({
    features: ['audio', 'maya', 'utt_id', 'spk_id'],
    num_rows: 2535
})

## Paso 1

In [7]:
import pandas as pd

raw_1h_df = pd.read_csv(RAW_DATA_1H)
raw_1h_df

,utt_id,maya,spanish,spk_id,start,end
0,c5EgkTbau2o_0000,baach,chachalaca,spk_001,00:00:22.550,00:00:24.450
1,c5EgkTbau2o_0001,chiich,abuela,spk_001,00:00:26.450,00:00:28.250
2,c5EgkTbau2o_0002,ch'íich',pájaro,spk_001,00:00:29.750,00:00:31.350
3,c5EgkTbau2o_0003,ja',agua,spk_001,00:00:32.450,00:00:33.550
4,c5EgkTbau2o_0004,kool,milpa,spk_001,00:00:35.350,00:00:37.050
...,...,...,...,...,...,...
1303,amigo_de_camacho_grabacion_17_0000,mixba'al,De nada.,spk_018,00:00:00.000,00:00:01.050
1304,amigo_de_camacho_grabacion_18_0000,ka anaktech jump'éel jats'uts k'iin,Que tengas un bonito día.,spk_018,00:00:00.000,00:00:02.322
1305,amigo_de_camacho_grabacion_19_0000,ka xi'iktech utsil,Que te vaya bien.,spk_018,00:00:00.000,00:00:01.588
1306,amigo_de_camacho_grabacion_20_0000,je'el k-ilikba'e',Nos vemos.,spk_018,00:00:00.000,00:00:01.489


In [8]:
# Paso 1 - Enriquecimiento del 1h (spk_001-018: YouTube + grabaciones propias)
#
# raw_1h_df["utt_id"] es el id CRUDO url-derivado (ej. c5EgkTbau2o_0000), que coincide
# con la columna "filename" del puente dataset.csv. Aqui solo extraemos spanish + timestamps;
# el utt_id spk-based y la maya canonica se asocian en el Paso 3 via el puente + HF.
enrich_1h = (
    raw_1h_df
    .rename(columns={"utt_id": "filename"})
    .loc[:, ["filename", "spanish", "start", "end"]]
    .copy()
)
print("enrich_1h:", enrich_1h.shape)
enrich_1h.head()

enrich_1h: (1308, 4)


,filename,spanish,start,end
0,c5EgkTbau2o_0000,chachalaca,00:00:22.550,00:00:24.450
1,c5EgkTbau2o_0001,abuela,00:00:26.450,00:00:28.250
2,c5EgkTbau2o_0002,pájaro,00:00:29.750,00:00:31.350
3,c5EgkTbau2o_0003,agua,00:00:32.450,00:00:33.550
4,c5EgkTbau2o_0004,milpa,00:00:35.350,00:00:37.050


## Paso 2

In [9]:
raw_3h_df = pd.read_csv(RAW_DATA_3H)
raw_3h_df

,audio_file,segment_id,time_start,time_end,duration,transcription
0,01_Anatolio_Pech,1,0.000,14.624,14.624,le tzicbalob nu kaaba'e xta'cun bixunan xta'cu...
1,01_Anatolio_Pech,2,14.624,25.312,10.688,a ta' kun vi xonaano le xonaano' jun p'ee co'l...
2,01_Anatolio_Pech,3,25.312,38.496,13.184,u kaaba'e' ya'ax che' paalo meeque ya'ala' ti'...
3,01_Anatolio_Pech,4,38.496,49.280,10.784,caan yila' t nupital u yaabitale' quyoco ti ch...
4,01_Anatolio_Pech,5,49.280,61.568,12.288,pero leti' utucul beyo le can wenequele u yumi...
...,...,...,...,...,...,...
1222,15_Mario_Chan,223,2246.016,2260.640,14.624,ca oce a manpure ca ada administracion pero ti...
1223,15_Mario_Chan,224,2260.640,2268.128,7.488,como toon un laj espricado too unbe taan too n...
1224,15_Mario_Chan,225,2268.128,2277.696,9.568,i le cus kaakpachkobe ma tu pakobi as ma tu pa...
1225,15_Mario_Chan,226,2277.696,2283.392,5.696,ten shamantiouye camiu nada siaa rosquiibiziq


In [10]:
# Paso 2 - Enriquecimiento del 3h (narraciones INALI, spk_019-032)
#
# El 3h NO tiene traduccion al espanol -> spanish = "".
# Los timestamps vienen en segundos (relativos al recording completo) y los pasamos a HH:MM:SS.mmm.
# El filename crudo se reconstruye igual que en el puente: audio_file + "_" + segment_id (SIN zero-padding).

def seconds_to_hhmmss(x: float) -> str:
    total_ms = int(round(float(x) * 1000))
    h, total_ms = divmod(total_ms, 3_600_000)
    m, total_ms = divmod(total_ms, 60_000)
    s, ms = divmod(total_ms, 1000)
    return f"{h:02}:{m:02}:{s:02}.{ms:03}"

raw_3h_df["filename"] = raw_3h_df["audio_file"] + "_" + raw_3h_df["segment_id"].astype(str)

enrich_3h = pd.DataFrame({
    "filename": raw_3h_df["filename"],
    "spanish": "",
    "start": raw_3h_df["time_start"].map(seconds_to_hhmmss),
    "end": raw_3h_df["time_end"].map(seconds_to_hhmmss),
})
print("enrich_3h:", enrich_3h.shape)
enrich_3h.head()

enrich_3h: (1227, 4)


,filename,spanish,start,end
0,01_Anatolio_Pech_1,,00:00:00.000,00:00:14.624
1,01_Anatolio_Pech_2,,00:00:14.624,00:00:25.312
2,01_Anatolio_Pech_3,,00:00:25.312,00:00:38.496
3,01_Anatolio_Pech_4,,00:00:38.496,00:00:49.280
4,01_Anatolio_Pech_5,,00:00:49.280,00:01:01.568


## Paso 3 - Construir el manifiesto de procedencia de audio

Este dataset documenta **solo de dónde viene el AUDIO** (no el texto, para evitar derechos de autor
sobre las transcripciones). Tomamos el dataset de HF como columna vertebral (`utt_id` spk-based + `spk_id`),
le asociamos los **timestamps** por el puente `dataset.csv`, y clasificamos cada segmento en tres grupos:

- **recordings** (grabaciones propias): lista plana; cada utterance es un wav en `data/recordings/{utt_id}.wav`.
- **youtube**: agrupado por video; `source` = URL de YouTube.
- **narraciones_mayas_campeche**: agrupado; `source` = URL pública del INALI.

El JSON final **no contiene texto** (maya/spanish).

In [11]:
# ---- Paso 3.1 - Columna vertebral: dataset HF (utt_id spk-based, spk_id) ----
# Accedemos por columna para NO decodificar el audio. (maya no se exporta: este
# manifiesto documenta solo la procedencia del AUDIO, no el texto.)
df = pd.DataFrame({
    "utt_id": ds["utt_id"],
    "spk_id": ds["spk_id"],
})

# ---- Paso 3.2 - Puente utt_id (spk) -> filename (id crudo) ----
bridge = pd.read_csv(BRIDGE_PATH)[["utt_id", "filename"]]
df = df.merge(bridge, on="utt_id", how="left")
assert df["filename"].notna().all(), "Hay utt_id del HF sin filename en el puente"

# ---- Paso 3.3 - Asociar timestamps (1h + 3h) por filename ----
enrich = pd.concat([enrich_1h, enrich_3h], ignore_index=True)[["filename", "start", "end"]]
df = df.merge(enrich, on="filename", how="left")

# ---- Paso 3.4 - Registro de fuentes: source_key -> (group, source, title) ----
def video_id_from_url(url: str) -> str:
    """YouTube -> id de 'v='; local -> carpeta_stem (para casar con el filename crudo)."""
    if url.startswith("http"):
        raw = url.split("v=")[1]
    else:
        p = Path(url)
        raw = f"{p.parent.name}_{p.stem}"
    return re.sub(r"[^A-Za-z0-9_-]+", "_", raw).strip("_")

with open(SEGMENTS_JSON_IN, encoding="utf-8") as f:
    curated = json.load(f)                      # spk_001-018 (YouTube + propias)

reg_rows = []
for s in curated:
    key = video_id_from_url(s["url"])
    if s["url"].startswith("http"):                             # YouTube
        reg_rows.append({"source_key": key, "group": "youtube",
                         "source": s["url"], "title": s["title"]})
    else:                                                       # grabaciones propias
        reg_rows.append({"source_key": key, "group": "recordings",
                         "source": "", "title": ""})
for af in sorted(raw_3h_df["audio_file"].unique()):             # narraciones INALI -> URL publica
    reg_rows.append({"source_key": af, "group": "narraciones_mayas_campeche",
                     "source": f"{INALI_BASE}/{af}.mp3", "title": af})
source_registry = pd.DataFrame(reg_rows).drop_duplicates("source_key")

# source_key = filename sin el sufijo "_<indice>"
df["source_key"] = df["filename"].str.replace(r"_\d+$", "", regex=True)
df = df.merge(source_registry, on="source_key", how="left")

print("df reconciliado:", df.shape, "| grupos:", df["group"].value_counts().to_dict())
df.head()

df reconciliado: (2535, 9) | grupos: {'narraciones_mayas_campeche': 1227, 'youtube': 1208, 'recordings': 100}


,utt_id,spk_id,filename,start,end,source_key,group,source,title
0,spk_001_utt_0001,spk_001,c5EgkTbau2o_0000,00:00:22.550,00:00:24.450,c5EgkTbau2o,youtube,https://www.youtube.com/watch?v=c5EgkTbau2o,"1.- Aprenda Maya (Alfabeto, Consonantes y Voca..."
1,spk_001_utt_0002,spk_001,c5EgkTbau2o_0001,00:00:26.450,00:00:28.250,c5EgkTbau2o,youtube,https://www.youtube.com/watch?v=c5EgkTbau2o,"1.- Aprenda Maya (Alfabeto, Consonantes y Voca..."
2,spk_001_utt_0003,spk_001,c5EgkTbau2o_0002,00:00:29.750,00:00:31.350,c5EgkTbau2o,youtube,https://www.youtube.com/watch?v=c5EgkTbau2o,"1.- Aprenda Maya (Alfabeto, Consonantes y Voca..."
3,spk_001_utt_0004,spk_001,c5EgkTbau2o_0003,00:00:32.450,00:00:33.550,c5EgkTbau2o,youtube,https://www.youtube.com/watch?v=c5EgkTbau2o,"1.- Aprenda Maya (Alfabeto, Consonantes y Voca..."
4,spk_001_utt_0005,spk_001,c5EgkTbau2o_0004,00:00:35.350,00:00:37.050,c5EgkTbau2o,youtube,https://www.youtube.com/watch?v=c5EgkTbau2o,"1.- Aprenda Maya (Alfabeto, Consonantes y Voca..."


In [12]:
# 3.5 Integridad (no deberian saltar)
assert df["group"].notna().all(), "hay filas sin grupo (source_key no mapeado)"
assert (df["start"].notna() & df["end"].notna()).all(), "faltan timestamps"
print("filas:", len(df), "| por grupo:", df["group"].value_counts().to_dict())

filas: 2535 | por grupo: {'narraciones_mayas_campeche': 1227, 'youtube': 1208, 'recordings': 100}


In [13]:
# 3.5b Procedencia de las grabaciones propias
#
# No vienen de ninguna fuente publica: son 1 utterance por wav y el propio
# corpus ES su fuente. Se apunta a data/final/audio/{utt_id}.wav, relativo a la
# raiz de datos, sin copiarlas a ningun sitio. El utt_id spk-based es anonimo
# por si mismo, asi que no expone identidad.
is_rec = df["group"] == "recordings"
df.loc[is_rec, "source"] = "final/audio/" + df.loc[is_rec, "utt_id"] + ".wav"

print(f"{int(is_rec.sum())} grabaciones propias -> {AUDIO}")

100 grabaciones propias -> /home/maucr/Documentos/thesis-mayan-ai/data/final/audio


In [14]:
# 3.6 Construir el manifiesto (3 grupos, SOLO procedencia de audio, sin texto) y exportar
out = {"recordings": [], "youtube": [], "narraciones_mayas_campeche": []}

# recordings: lista plana, un audio por utterance
rec = df[df["group"] == "recordings"].sort_values("utt_id")
out["recordings"] = [
    {"utt_id": r.utt_id, "start": r.start, "end": r.end, "spk_id": r.spk_id, "source": r.source}
    for r in rec.itertuples(index=False)
]

# youtube y narraciones: agrupados por fuente (source + title), con timestamps
for grp in ["youtube", "narraciones_mayas_campeche"]:
    g = df[df["group"] == grp].sort_values(["source", "start"], kind="stable")
    for (source, title), sub in g.groupby(["source", "title"], sort=False):
        out[grp].append({
            "source": source,
            "title": title,
            "segments": [
                {"utt_id": r.utt_id, "start": r.start, "end": r.end, "spk_id": r.spk_id}
                for r in sub.itertuples(index=False)
            ],
        })

with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=4)
print("Escrito", OUT_JSON, "->", {k: len(v) for k, v in out.items()})

Escrito /home/maucr/Documentos/thesis-mayan-ai/data/final/manifests/source_segments.json -> {'recordings': 100, 'youtube': 44, 'narraciones_mayas_campeche': 15}


In [15]:
# 3.7 Validacion del manifiesto
d = json.load(open(OUT_JSON, encoding="utf-8"))
assert set(d) == {"recordings", "youtube", "narraciones_mayas_campeche"}

total = len(d["recordings"]) + sum(len(x["segments"]) for x in d["youtube"]) \
        + sum(len(x["segments"]) for x in d["narraciones_mayas_campeche"])
assert total == len(df), f"utt_id {total} != df {len(df)}"

raw = open(OUT_JSON, encoding="utf-8").read()
assert '"maya"' not in raw and '"spanish"' not in raw, "quedo texto en el JSON"

missing = [r["source"] for r in d["recordings"] if not (DATA / r["source"]).exists()]
assert not missing, f"faltan audios propios: {missing[:3]}"

print("OK:", {k: len(v) for k, v in d.items()}, "| total utt_id:", total, "| sin texto")

OK: {'recordings': 100, 'youtube': 44, 'narraciones_mayas_campeche': 15} | total utt_id: 2535 | sin texto


In [16]:
# 3.8 OPCIONAL - Actualizar el dataset HF con las columnas nuevas (spanish, start, end, url)
#
# add_column preserva el feature Audio existente (no re-castea el audio).
# Descomenta para publicar. Requiere notebook_login() con token de escritura.

# ds_enriched = ds
# for col in ["spanish", "start", "end", "url"]:
#     m = dict(zip(df["utt_id"], df[col]))
#     ds_enriched = ds_enriched.add_column(col, [m.get(u, "") for u in ds["utt_id"]])
# ds_enriched.push_to_hub(REPO_ID, private=True)
# ds_enriched